# Reproducible research figures

This notebook **generates** figures from the checked-in experimental JSON results and synthetic-scene records. It does **not** train the models or recompute metrics. Figures are written under `outputs/dataset/` or `outputs/experiments/<experiment>/` according to the subject. The figures used in the Vietnamese report are additionally copied into `report/Figures/`.

**Sources of truth:** metric JSON files and scene/counterfactual records, not values entered by hand. The pipeline diagrams are explanatory schematics generated with Matplotlib and are *not* learned attention maps. Existing research papers under `thesis/` are never modified.

Run **Restart & Run All** in VS Code/Jupyter from any working directory. `PROJECT_ROOT` is determined using the notebook location or verified parent markers. No imports from research training modules are needed for metric charts.

In [ ]:
from pathlib import Path
import json
import math
import shutil
import sys
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch, Circle, Rectangle, Polygon
from PIL import Image, ImageDraw, ImageFont

# Notebook path resolution: current working directory may be repository root,
# notebooks/, or a VS Code/Jupyter launch directory.
# If using a different directory, set PROJECT_ROOT explicitly to the repository folder.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path.cwd() / 'visual_world', Path.cwd() / 'aio_m5_visual_world']:
    if (candidate / 'pyproject.toml').is_file() and (candidate / 'outputs' / 'dataset' / 'train.jsonl').is_file():
        PROJECT_ROOT = candidate.resolve()
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find project root: open this repository in VS Code or set PROJECT_ROOT to its absolute path.')
DATASET = PROJECT_ROOT / 'outputs' / 'dataset'
EXPERIMENTS = PROJECT_ROOT / 'outputs' / 'experiments'
REPORT_FIGURES = PROJECT_ROOT / 'report' / 'Figures'
REPORT_FIGURES.mkdir(parents=True, exist_ok=True)
for p in (PROJECT_ROOT, PROJECT_ROOT / 'src'):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 180, 'axes.spines.top': False,
                     'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': 0.18,
                     'font.family': 'DejaVu Sans', 'font.size': 10})

GENERATED = []

def load_json(path):
    if not path.is_file():
        raise FileNotFoundError(f'Missing recorded experiment output: {path}')
    return json.loads(path.read_text(encoding='utf-8'))

def save_figure(fig, group, filename, report_filename=None):
    folder = (DATASET if group == 'dataset' else EXPERIMENTS / group)
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / filename
    fig.savefig(target, dpi=180, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    GENERATED.append(target)
    if report_filename:
        shutil.copy2(target, REPORT_FIGURES / report_filename)
    print(target.relative_to(PROJECT_ROOT))
    return target

def export_picture(img, group, filename, report_filename=None):
    folder = (DATASET if group == 'dataset' else EXPERIMENTS / group)
    folder.mkdir(parents=True, exist_ok=True)
    target = folder / filename
    img.save(target)
    GENERATED.append(target)
    if report_filename:
        shutil.copy2(target, REPORT_FIGURES / report_filename)
    print(target.relative_to(PROJECT_ROOT))
    return target

## 1. Model learning curves and grounding controls

In [ ]:
original = load_json(EXPERIMENTS / 'visual_grounding_debug' / 'color_diagnostic.json')
color = load_json(EXPERIMENTS / 'color_grounding' / 'grounding_diagnostic.json')
shape = load_json(EXPERIMENTS / 'attribute_binding' / 'shape_diagnostic.json')
pair = load_json(EXPERIMENTS / 'attribute_binding' / 'pair_diagnostic.json')

# Original ViT versus constant-image control, directly from recorded accuracy history.
fig, ax = plt.subplots(figsize=(8.6, 3.6))
for mode, color_code in [('full', '#29598a'), ('text-only', '#be7842')]:
    h = original['runs']['template'][mode]['history']
    ax.plot([r['epoch'] for r in h], [100*r['id_dev_accuracy'] for r in h],
            color=color_code, label='Ảnh + câu hỏi' if mode=='full' else 'Chỉ câu hỏi')
ax.set(xlabel='Epoch', ylabel='Accuracy ID-dev (%)', title='Mô hình ban đầu: phát hiện màu')
ax.set_ylim(0, 105); ax.legend(loc='upper right')
save_figure(fig, 'visual_grounding_debug', 'original_model_color_accuracy.png')

# Revised visual grounding model.
fig, ax = plt.subplots(figsize=(8.6, 3.6))
h = color['runs']['full']['history']
ax.plot([r['epoch'] for r in h], [100*r['train_accuracy'] for r in h],
        label='Train', linewidth=2)
ax.plot([r['epoch'] for r in h], [100*r['id_dev_accuracy'] for r in h],
        label='ID-dev', linewidth=2)
ax.axhline(50, color='gray', alpha=.6, linestyle=':', label='Mốc cân bằng 50%')
ax.set(xlabel='Epoch', ylabel='Accuracy (%)', ylim=(40,105),title='Nhận biết màu từ ảnh: kết quả huấn luyện')
ax.legend(loc='lower right')
save_figure(fig, 'color_grounding', 'color_learning_accuracy.png', 'color_learning_accuracy.png')

# Control comparison (read directly from the saved experiment outputs).
fig, ax = plt.subplots(figsize=(7.7, 3.5))
values = [100*color['runs']['full']['final']['id_dev_accuracy'],
          100*color['runs']['text-only']['final']['id_dev_accuracy'],
          100*color['runs']['full']['constant_image_dev_accuracy']]
labels = ['Ảnh thật + câu hỏi', 'Huấn luyện chỉ câu hỏi', 'Thay ảnh bằng hằng số']
bars=ax.bar(labels, values, color=['#357caa','#96a3af','#cc9672'])
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set(ylabel='Accuracy ID-dev (%)', title='Đối chứng để kiểm tra phụ thuộc vào ảnh',ylim=(0,112))
ax.tick_params(axis='x',labelrotation=8)
save_figure(fig, 'color_grounding', 'color_grounding_controls.png')

outputs\experiments\visual_grounding_debug\original_model_color_accuracy.png
outputs\experiments\color_grounding\color_learning_accuracy.png
outputs\experiments\color_grounding\color_grounding_controls.png


WindowsPath('C:/Users/Quoc Thai/Downloads/AI/AIO/Conquer/AIO Conquer 5 - visual_world/outputs/experiments/color_grounding/color_grounding_controls.png')

## 2. Shape recognition, attribute binding, and generalization controls

In [ ]:
fig, ax = plt.subplots(figsize=(8.6, 3.6))
for name, d, c in [('Hình dạng',shape,'#2979a5'), ('Ghép màu–hình',pair,'#bf824e')]:
    h=d['runs']['full']['history']
    ax.plot([r['epoch'] for r in h], [100*r['id_dev_accuracy'] for r in h],label=name,color=c,linewidth=2)
ax.set(xlabel='Epoch',ylabel='Accuracy ID-dev (%)',title='Nhận biết hình dạng và ghép thuộc tính',ylim=(40,100))
ax.legend(loc='lower right')
save_figure(fig,'attribute_binding','attribute_accuracy_curves.png')

fig, ax = plt.subplots(figsize=(8.6, 3.6))
h=pair['runs']['full']['history']
ax.plot([r['epoch'] for r in h],[r['train_loss'] for r in h],label='Train',linewidth=2)
ax.plot([r['epoch'] for r in h],[r['id_dev_loss'] for r in h],label='ID-dev',linewidth=2)
ax.set(xlabel='Epoch',ylabel='Cross-entropy',title='Ghép màu–hình: khoảng cách giữa train và ID-dev')
ax.legend(loc='upper left')
save_figure(fig,'attribute_binding','attribute_binding_dev_loss.png','attribute_binding_dev_loss.png')

controls={name:load_json(EXPERIMENTS/'generalization_controls'/f'{name}_generalization.json') for name in ('shape','pair')}
variants=['baseline','augment','decay']
fig, ax=plt.subplots(figsize=(8.6,3.8))
x=np.arange(len(variants));w=.35
for j,(task,label,col) in enumerate([('shape','Hình dạng','#3c78a5'),('pair','Ghép thuộc tính','#c28854')]):
    values=[100*next(r for r in controls[task]['runs'] if r['variant']==v)['best_checkpoint']['id_dev']['accuracy'] for v in variants]
    bars=ax.bar(x+(j-.5)*w,values,w,label=label,color=col)
    ax.bar_label(bars,fmt='%.1f',padding=3,fontsize=9)
ax.set(xticks=x,xticklabels=['Gốc','Tăng cường ảnh','Weight decay'],ylabel='Accuracy ID-dev (%)',ylim=(0,100),title='Độ chính xác tại checkpoint có ID-dev loss thấp nhất')
ax.legend(loc='upper right')
save_figure(fig,'generalization_controls','regularization_accuracy_comparison.png','regularization_accuracy_comparison.png')

fig, ax=plt.subplots(figsize=(8.6,3.6))
for task,col in [('shape','#3976a1'),('pair','#c58b50')]:
    values=[next(r for r in controls[task]['runs'] if r['variant']==v)['best_checkpoint']['paired_counterfactual']['prediction_flip_rate']*100 for v in variants]
    ax.plot(variants,values,marker='o',linewidth=2,label=task,color=col)
ax.set_xticks(range(len(variants)), ['Gốc','Tăng cường','Weight decay'])
ax.set(ylabel='Prediction flip rate (%)',ylim=(0,100),title='Độ nhạy với phép tráo ảnh giữa các câu hỏi giống nhau')
ax.legend()
save_figure(fig,'generalization_controls','image_swap_sensitivity.png')

outputs\experiments\attribute_binding\attribute_accuracy_curves.png
outputs\experiments\attribute_binding\attribute_binding_dev_loss.png
outputs\experiments\generalization_controls\regularization_accuracy_comparison.png
outputs\experiments\generalization_controls\image_swap_sensitivity.png


WindowsPath('C:/Users/Quoc Thai/Downloads/AI/AIO/Conquer/AIO Conquer 5 - visual_world/outputs/experiments/generalization_controls/image_swap_sensitivity.png')

## 3. Matched-exposure learning-order results

In [ ]:
learning=load_json(EXPERIMENTS/'learning_order'/'schedule_comparison.json')
runs={r['schedule']:r for r in learning['runs']}
assert set(runs)=={'mixed','staged','reverse'}
assert {r['steps_run'] for r in runs.values()}=={450}
assert {r['exposure_plan']['repetitions_per_example'] for r in runs.values()}=={24}
labels={'mixed':'Trộn','staged':'Theo giai đoạn','reverse':'Đảo ngược'}
colors={'color':'#3177aa','shape':'#629a7a','pair':'#c99156'}

fig,ax=plt.subplots(figsize=(8.6,3.8));x=np.arange(3);w=.26
for i,task in enumerate(('color','shape','pair')):
    values=[100*runs[s]['final']['id_dev']['by_template'][task] for s in runs]
    bars=ax.bar(x+(i-1)*w,values,width=w,label={'color':'Màu','shape':'Hình','pair':'Ghép thuộc tính'}[task],color=colors[task])
    ax.bar_label(bars,fmt='%.1f',padding=2,fontsize=8)
ax.set(xticks=x,xticklabels=[labels[s] for s in runs],ylabel='Accuracy ID-dev (%)',ylim=(0,112),title='Kết quả cuối ở bước 450: cùng số ví dụ và bước cập nhật')
ax.legend(loc='upper center',bbox_to_anchor=(.5,-.12),ncol=3,frameon=False)
save_figure(fig,'learning_order','learning_schedule_final_accuracy.png','learning_schedule_final_accuracy.png')

fig, ax=plt.subplots(figsize=(8.6,3.9))
for schedule,c in [('mixed','#3978a6'),('staged','#46946d'),('reverse','#bf8657')]:
    h=runs[schedule]['history']
    ax.plot([snap['step'] for snap in h], [100*snap['id_dev']['by_template']['pair'] for snap in h],
            marker='o', markersize=4, linewidth=2,label=labels[schedule],color=c)
ax.axvline(375,linestyle=':',color='gray',alpha=.7)
ax.text(379,46,'Bắt đầu ôn tập chung',fontsize=8,color='gray')
ax.set(xlabel='Bước cập nhật',ylabel='Accuracy ghép thuộc tính ID-dev (%)',ylim=(40,100),title='Diễn biến học và duy trì khả năng ghép thuộc tính')
ax.legend(loc='upper left')
save_figure(fig,'learning_order','learning_schedule_binding_trajectories.png','learning_schedule_binding_trajectories.png')

fig,ax=plt.subplots(figsize=(7.6,3.6))
values=[100*runs[s]['paired_counterfactual']['prediction_flip_rate'] for s in runs]
bars=ax.bar([labels[s] for s in runs], values, color=['#3978a6','#46946d','#bf8657'])
ax.bar_label(bars,fmt='%.0f%%',padding=3)
ax.set(ylabel='Prediction flip rate (%)',ylim=(0,100),title='Nhạy cảm với tráo ảnh ở tác vụ ghép thuộc tính')
save_figure(fig,'learning_order','binding_image_swap_sensitivity.png')

outputs\experiments\learning_order\learning_schedule_final_accuracy.png
outputs\experiments\learning_order\learning_schedule_binding_trajectories.png
outputs\experiments\learning_order\binding_image_swap_sensitivity.png


WindowsPath('C:/Users/Quoc Thai/Downloads/AI/AIO/Conquer/AIO Conquer 5 - visual_world/outputs/experiments/learning_order/binding_image_swap_sensitivity.png')

## 4. Dataset gallery and verified counterfactual examples

In [ ]:
# Regenerate images from stored scene specifications, without using any test
# or OOD labels to tune model parameters. Gallery uses training rows only.
from src.visual_world.generate import read_rows
from src.visual_world.gallery import tile, W, H
from src.visual_world.render import render
from src.visual_world.world import from_dict

train_rows=list(read_rows(DATASET/'train.jsonl'))
chosen=[]
for template in ('color','shape','pair','left','above','xor'):
    for target in (0,1):
        selected=[r for r in train_rows if r['template']==template and r['label']==target]
        chosen.extend(selected[:2])
assert len(chosen)==24
sheet=Image.new('RGB',(W*4,H*6),(241,244,244))
for i,row in enumerate(chosen):
    sheet.paste(tile(row),((i%4)*W,(i//4)*H))
export_picture(sheet,'dataset','synthetic_scene_gallery.png','synthetic_scene_gallery.png')

# Counterfactual metadata was already recorded with paired scenes. Rendering it
# does not regenerate or change labels and does not require loading held-out splits.
examples=load_json(DATASET/'counterfactuals.json')
assert len(examples)==8
paired=Image.new('RGB',(W*4,H*4),(241,244,244))
for i,record in enumerate(examples):
    a={'scene':record['before_scene'],'split':record['split_source'],
       'template':record['template'],'label':record['original_label'],
       'id':record['original_id'],'query_tokens':record['query_tokens']}
    b={**a,'scene':record['after_scene'],'label':record['new_label'],
       'id':record['intervened_graph_id']}
    paired.paste(tile(a),((2*i%4)*W,(i//2)*H))
    paired.paste(tile(b),(((2*i+1)%4)*W,(i//2)*H))
export_picture(paired,'dataset','scene_counterfactual_examples.png','scene_counterfactual_examples.png')

outputs\dataset\synthetic_scene_gallery.png
outputs\dataset\scene_counterfactual_examples.png


WindowsPath('C:/Users/Quoc Thai/Downloads/AI/AIO/Conquer/AIO Conquer 5 - visual_world/outputs/dataset/scene_counterfactual_examples.png')

## 5. Dataset distribution and explanatory diagrams (schematics)

In [ ]:
# Dataset-distribution figure: count observed yellow-object prevalence.
id_dev=list(read_rows(DATASET/'id_dev.jsonl'))
ood_dev=list(read_rows(DATASET/'ood_dev.jsonl'))
def yellow_fraction(rows, task='pair'):
    sub=[r for r in rows if r['template']==task]
    yes=sum(any(obj['color']=='yellow' for obj in r['scene'].values() if isinstance(obj,dict) and 'color' in obj) for r in sub)
    return yes,len(sub)
counts=[yellow_fraction(id_dev),yellow_fraction(ood_dev)]
fig,ax=plt.subplots(figsize=(7.0,3.5))
bars=ax.bar(['ID-dev: ghép thuộc tính','OOD-dev: ghép thuộc tính'], [100*a/b for a,b in counts], color=['#c69e51','#5c89aa'])
for rect,(n,total) in zip(bars,counts):ax.text(rect.get_x()+rect.get_width()/2,rect.get_height()+2,f'{n}/{total}',ha='center')
ax.set(ylabel='Cảnh chứa vàng (%)',ylim=(0,115),title='Kiểm toán màu vàng trên tập đánh giá')
save_figure(fig,'dataset','yellow_presence_distribution.png')

# A minimal reusable flow-diagram renderer. Diagrams are architectural 
# explanations, not outputs of a trained network.
def diagram(title,stages,group,filename,subtitle=None):
    fig,ax=plt.subplots(figsize=(8.6,max(3.2,0.84*len(stages)+1.1)))
    ax.set_xlim(0,10);ax.set_ylim(0,len(stages)+.8);ax.axis('off')
    ax.text(5,len(stages)+.57,title,ha='center',va='center',fontsize=13,fontweight='bold')
    for i,item in enumerate(stages):
        y=len(stages)-i-.45
        p=FancyBboxPatch((1.05,y-.26),7.9,.52,boxstyle='round,pad=0.035,rounding_size=0.11',
                         facecolor='#e7edf3' if i%2==0 else '#dce9e6',edgecolor='#b5c4cf',lw=1.0)
        ax.add_patch(p)
        ax.text(5,y,item,ha='center',va='center',fontsize=10)
        if i<len(stages)-1:
            ax.annotate('',xy=(5,y-.63),xytext=(5,y-.29),arrowprops=dict(arrowstyle='-|>',lw=1.2,color='#61768a'))
    if subtitle:fig.text(.5,.018,subtitle,ha='center',fontsize=8,color='#556779')
    save_figure(fig,group,filename)

diagram('Tạo dữ liệu cảnh tổng hợp',[
    'Chọn màu, hình dạng, kích cỡ và cấu trúc vật thể',
    'Sinh đồ thị cảnh hợp lệ; ràng buộc tổ hợp ID/OOD',
    'Tạo truy vấn logic và nhãn bằng symbolic oracle',
    'Tách manifest, lưu SHA-256 và kiểm toán',
    'Xuất ảnh 64×64 cùng câu hỏi cho mô hình'],
    'dataset','dataset_generation_flow.png')

diagram('Pipeline render ảnh',[
    'Đồ thị cảnh: ba vật thể và tọa độ',
    'Vẽ hình màu lên canvas RGB 256×256',
    'Khử răng cưa bằng Lanczos, thu ảnh 64×64',
    'Chuẩn hóa tensor (3, 64, 64) với giá trị trong [0,1]'],
    'dataset','image_rendering_flow.png')

diagram('Đường đi dữ liệu của ViT gốc',[
    'Ảnh → 64 patch 8×8 + visual CLS',
    'Visual Transformer: 65 token',
    'Câu hỏi + question CLS → Question Transformer',
    'Cross-attention: query token đọc 64 patch, loại visual CLS trực tiếp',
    'Fusion Transformer → fused question CLS',
    'Phân loại True / False'],
    'transformer_baseline','baseline_architecture.png')

diagram('Tham chiếu kiến trúc Vision Transformer',[
    'Ảnh đầu vào',
    'Patch embedding (không phải phát hiện đối tượng sẵn)',
    'Thêm positional embedding',
    'Self-attention học biểu diễn theo ngữ cảnh',
    'Truy vấn thông tin theo câu hỏi',
    'Đầu phân loại và loss'],
    'transformer_baseline','vision_transformer_overview.png')

diagram('Mã hóa câu hỏi cho mô hình',[
    'EXISTS_PAIR  red  circle',
    '[CLS]  EXISTS_PAIR  red  circle  [PAD] ...',
    'Tra từ điển token → chuỗi ID dài 32',
    'Embedding học được, dùng để truy vấn patch ảnh'],
    'transformer_baseline','question_token_encoding.png')

diagram('Ghép thuộc tính trên cùng vị trí ảnh',[
    'Ảnh → 256 patch 4×4 + ViT và nhánh residual',
    'Câu hỏi → embedding toán tử, thuộc tính 1 và thuộc tính 2',
    'Tính s1, s2, s_joint cho từng patch',
    'Tính tương tác tanh(s1) × tanh(s2) tại cùng patch',
    'Max + mean của 4 nhánh → 8 số → MLP True / False'],
    'attribute_binding','attribute_binding_pipeline.png')

diagram('Các phép đo trong nghiên cứu thứ tự học',[
    'Accuracy và cross-entropy trên ID-dev (theo từng tác vụ)',
    'Đường học theo bước và ranh giới các phase',
    'Đối chứng constant-image và matched-question image swap',
    'So sánh cùng seed, cùng ví dụ, cùng cập nhật',
    'OOD và thích nghi: mục tiêu tương lai, chưa kiểm chứng ở đây'],
    'learning_order','research_measurements.png')

outputs\dataset\yellow_presence_distribution.png
outputs\dataset\dataset_generation_flow.png
outputs\dataset\image_rendering_flow.png
outputs\experiments\transformer_baseline\baseline_architecture.png
outputs\experiments\transformer_baseline\vision_transformer_overview.png
outputs\experiments\transformer_baseline\question_token_encoding.png
outputs\experiments\attribute_binding\attribute_binding_pipeline.png
outputs\experiments\learning_order\research_measurements.png


## 6. Concrete illustration: independent attribute detection can be misleading

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(8.8,3.3))
for ax,present in zip(axes,[True,False]):
    ax.set(xlim=(0,5),ylim=(0,3));ax.set_aspect('equal');ax.axis('off')
    ax.add_patch(Rectangle((0,0),5,3,facecolor='#f2f4f6',edgecolor='none'))
    ax.add_patch(Circle((1.15,2),.46,facecolor='#477db1',edgecolor='#2b3744',lw=1.2))
    ax.add_patch(Rectangle((2.65,1.65),.85,.85,facecolor='#477db1' if present else '#cc5e56',edgecolor='#2b3744',lw=1.2))
    ax.text(2.5,.5,'Có hình vuông màu xanh?' if present else 'Có hình vuông màu xanh?',ha='center',fontsize=10)
    ax.set_title('Đúng: có cùng vật thể' if present else 'Sai: thuộc tính nằm ở vật thể khác',fontsize=10)
fig.suptitle('Nhận biết màu và hình độc lập không bảo đảm ghép đúng thuộc tính',fontsize=12)
save_figure(fig,'attribute_binding','attribute_binding_counterexample.png')

# File manifest for graphics created during this run.
manifest={'generator':'notebooks/generate_research_figures.ipynb',
          'note':'Charts from stored JSON. Galleries from scene data. Diagrams are explanatory schematics.',
          'files':[str(p.relative_to(PROJECT_ROOT)) for p in GENERATED]}
manifest_path=EXPERIMENTS/'figure_manifest.json'
manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+chr(10),encoding='utf-8')
print(f'Generated {len(GENERATED)} PNG assets; manifest: {manifest_path.relative_to(PROJECT_ROOT)}')

outputs\experiments\attribute_binding\attribute_binding_counterexample.png
Generated 21 PNG assets; manifest: outputs\experiments\figure_manifest.json
